# Retail Operations Intelligence EDA

A full-stack business analyst project that goes from messy operational data to dashboard-ready outputs for Tableau.

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
plt.rcParams["figure.figsize"] = (10, 5)


from dateutil import parser
def mixed_parse(val):
    if pd.isna(val):
        return pd.NaT
    s = str(val).strip()
    if not s:
        return pd.NaT
    for dayfirst in (False, True):
        try:
            return parser.parse(s, dayfirst=dayfirst)
        except Exception:
            continue
    return pd.NaT


## 2. Load raw data

In [ ]:
orders = pd.read_csv("../data/raw/orders.csv")
customers = pd.read_csv("../data/raw/customers.csv")
print("orders:", orders.shape, "customers:", customers.shape)
display(orders.head())
display(customers.head())


## 3. Data quality audit

In [ ]:
def audit(df):
    return pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing_pct": (df.isna().mean()*100).round(2),
        "n_unique": df.nunique(dropna=False)
    }).sort_values("missing_pct", ascending=False)

display(audit(orders))
display(audit(customers))
print("Duplicate orders:", orders.duplicated().sum())
print("Duplicate customers:", customers["customer_id"].duplicated().sum())


## 4. Clean and standardize

In [ ]:
orders["order_timestamp"] = orders["order_timestamp"].apply(mixed_parse)
orders["promised_delivery_date"] = orders["promised_delivery_date"].apply(mixed_parse)
orders["actual_delivery_date"] = orders["actual_delivery_date"].apply(mixed_parse)
customers["first_order_date"] = customers["first_order_date"].apply(mixed_parse)

num_cols = ["quantity","gross_sales_usd","discount_rate","net_sales_usd","cost_usd","satisfaction_score"]
for col in num_cols:
    orders[col] = pd.to_numeric(orders[col], errors="coerce")

orders.loc[orders["net_sales_usd"] < 0, "net_sales_usd"] = np.nan
orders["quantity"] = orders["quantity"].fillna(1)
orders = orders.dropna(subset=["order_id","customer_id","order_timestamp"]).drop_duplicates(subset=["order_id"], keep="first")
customers = customers.dropna(subset=["customer_id"]).drop_duplicates(subset=["customer_id"], keep="first")

## 5. Feature engineering

In [ ]:
orders["delivery_delay_days"] = (orders["actual_delivery_date"] - orders["promised_delivery_date"]).dt.days
orders["late_flag"] = (orders["delivery_delay_days"] > 0).astype(int)
orders["gross_margin_usd"] = orders["net_sales_usd"] - orders["cost_usd"]
orders["gross_margin_pct"] = orders["gross_margin_usd"] / orders["net_sales_usd"]
orders["order_month"] = orders["order_timestamp"].dt.to_period("M").astype(str)

df = orders.merge(customers, on="customer_id", how="left")
display(df.head())


## 6. Advanced EDA

In [ ]:
kpis = {
    "net_sales": round(df["net_sales_usd"].sum(), 2),
    "gross_margin": round(df["gross_margin_usd"].sum(), 2),
    "gross_margin_pct": round(df["gross_margin_usd"].sum() / df["net_sales_usd"].sum(), 4),
    "return_rate": round(df["returned_flag"].mean(), 4),
    "late_delivery_rate": round(df["late_flag"].mean(), 4),
    "avg_satisfaction": round(df["satisfaction_score"].mean(), 2),
}
kpis


In [ ]:
monthly = df.groupby("order_month")[["net_sales_usd","gross_margin_usd"]].sum()
monthly.plot(title="Monthly sales and margin")
plt.ylabel("USD")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

category = (df.groupby("category")
              .agg(net_sales_usd=("net_sales_usd","sum"),
                   gross_margin_pct=("gross_margin_pct","mean"),
                   return_rate=("returned_flag","mean"),
                   avg_satisfaction=("satisfaction_score","mean"),
                   orders=("order_id","nunique"))
              .sort_values("net_sales_usd", ascending=False))
display(category)


In [ ]:
channel_seg = (df.groupby(["sales_channel","customer_segment"])
                 .agg(net_sales_usd=("net_sales_usd","sum"),
                      gross_margin_pct=("gross_margin_pct","mean"),
                      late_delivery_rate=("late_flag","mean"),
                      return_rate=("returned_flag","mean"))
                 .round(4))
display(channel_seg)


In [ ]:
ops = (df.groupby(["warehouse","shipping_mode"])
         .agg(orders=("order_id","nunique"),
              late_delivery_rate=("late_flag","mean"),
              avg_delay_days=("delivery_delay_days","mean"),
              avg_satisfaction=("satisfaction_score","mean"))
         .sort_values("late_delivery_rate", ascending=False))
display(ops)


## 7. Export clean tables for dashboarding

In [ ]:
clean_orders = df.copy()
clean_orders.to_csv("../data/clean/clean_orders_enriched.csv", index=False)

dashboard_scorecard = (df.groupby(["order_month","sales_region","sales_channel","customer_segment"])
                        .agg(orders=("order_id","nunique"),
                             customers=("customer_id","nunique"),
                             net_sales_usd=("net_sales_usd","sum"),
                             gross_margin_usd=("gross_margin_usd","sum"),
                             gross_margin_pct=("gross_margin_pct","mean"),
                             return_rate=("returned_flag","mean"),
                             late_delivery_rate=("late_flag","mean"),
                             avg_satisfaction=("satisfaction_score","mean"))
                        .reset_index())
dashboard_scorecard.to_csv("../data/clean/dashboard_scorecard.csv", index=False)

print("Exports created.")


## 8. Tableau handoff notes

After running this notebook:
1. connect Tableau to `data/clean/dashboard_scorecard.csv` and `data/clean/clean_orders_enriched.csv`
2. create the KPI cards and sheets described in `dashboard/tableau_dashboard_spec.md`
3. add the calculated fields listed in `dashboard/calculated_fields.csv`
4. publish a screenshot of the final dashboard to GitHub for portfolio presentation
